# Personal Project – Understand → Generate → Align
*Exploratory notebook created by Brian – ~2–3 hours of work, run on Colab / Kaggle T4*

---

Three questions. Three acts. One spine.

| Act | Question it answers | Model | Paradigm |
|---|---|---|---|
| 1. Understand | How do we represent meaning? | BERT (tiny) | pretrain → finetune |
| 2. Generate | What if the task is to *produce* text, not classify it? | PGN | encoder–decoder + copy |


Three threads tie everything together:
1. **The pretrain→finetune pattern** — named explicitly each time it reappears.
2. **Teacher forcing** — introduced before Act 2, used to train PGN, revealed as SFT in Act 3.


## Before you run this notebook

**Colab / Kaggle setup:**
1. **Internet on** — required for model/dataset downloads (Settings → Internet → On).
2. **GPU = T4** for Act 3 (Settings → Accelerator → GPU T4). Acts 1–2 are CPU-friendly.
3. Kaggle GPU quota is 30 h/week — don't leave sessions running overnight.

Acts 1 and 2 run entirely on CPU. Act 3 (Qwen GRPO) needs a T4 GPU — a warning cell will remind you.

In [ ]:
# Pin versions for reproducibility. Do NOT reinstall torch — runtime ships the CUDA build.
!pip install -q transformers tokenizers datasets evaluate rouge_score matplotlib

In [ ]:
import random, math, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

# Acts 1-2 Auto* imports
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Heavy Act-3 imports are deferred to the Act 3 setup cell.

## YOUR TO-DO LIST (live coding)
While the instructor codes, you fill these blanks (goal + equation hint only).

Act 1 — BERT
[ ] 1. SentimentModel head-swap: attach a fresh nn.Linear head onto the pretrained
       DistilBERT body; forward returns head(last_hidden_state[:, 0]).

Act 2 — PGN (one cell, 4 pieces)
[ ] 2. PIECE 1 Attention (eqs 1-3): scores e, a=softmax(e), context ctx.
[ ] 3. PIECE 2 Vocab dist (eq 4): P_vocab = softmax(Vproj([s_t; ctx])).
[ ] 4. PIECE 3 p_gen gate (eq 8): sigmoid(w · [ctx; s_t; x]).
[ ] 5. PIECE 4 Mixed final dist (eq 9): generate mass into first V cols, then
       scatter-add copy mass (1-p_gen)*a onto src_ext columns. (THE CLIMAX.)

The three debug-together bugs (tokenizer mismatch, scatter-add, GRPO ratio≡1)
are NOT blanks — the instructor drives those.

## Act 1 — Understand

### 1a. BPE Tokenization

**Anchor question:** What happens to a word the tokenizer has never seen?

We train a Byte-Pair Encoding tokenizer on a ~1 MB Wikipedia subset and watch it build a subword vocabulary. The answer to the anchor question foreshadows the OOV problem that Act 2's Pointer-Generator Network is designed to solve.

In [ ]:
# Load a small wikitext subset (~1 MB of text) for BPE training.
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
texts = [row["text"] for row in wiki if len(row["text"].strip()) > 0]
print(f"Loaded {len(texts):,} non-empty lines, ~{sum(len(t) for t in texts)//1024} KB of text")

with open("/tmp/wiki_train.txt", "w") as f:
    f.write("\n".join(texts))

Loaded 23,767 non-empty lines, ~10637 KB of text


In [ ]:
# Train a BPE tokenizer with a ~6k token vocabulary.
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=6000, min_frequency=2,
                     special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
bpe_tokenizer.train(files=["/tmp/wiki_train.txt"], trainer=trainer)
print(f"Vocabulary size: {bpe_tokenizer.get_vocab_size()}")

Vocabulary size: 6000


In [ ]:
# Inspect learned subword pieces and tokenize sample sentences.
vocab = bpe_tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
print("Sample vocab entries (subword pieces):")
for piece, idx in sorted_vocab[5:25]:
    print(f"  {idx:5d}: {piece!r}")

print()
samples = [
    "the quick brown fox jumps over the lazy dog",
    "transformer models use self-attention mechanisms",
    "Zxyqvr is a rare entity that BPE has never seen",
]
for s in samples:
    enc = bpe_tokenizer.encode(s)
    print(f"Input:  {s!r}")
    print(f"Tokens: {enc.tokens}")
    print()

Sample vocab entries (subword pieces):
      5: '!'
      6: '"'
      7: '#'
      8: '$'
      9: '%'
     10: '&'
     11: "'"
     12: '('
     13: ')'
     14: '*'
     15: '+'
     16: ','
     17: '-'
     18: '.'
     19: '/'
     20: '0'
     21: '1'
     22: '2'
     23: '3'
     24: '4'

Input:  'the quick brown fox jumps over the lazy dog'
Tokens: ['the', 'quick', 'brown', 'fox', 'j', 'um', 'ps', 'over', 'the', 'l', 'az', 'y', 'dog']

Input:  'transformer models use self-attention mechanisms'
Tokens: ['trans', 'former', 'mode', 'ls', 'use', 'self', '-', 'attention', 'mechan', 'is', 'ms']

Input:  'Zxyqvr is a rare entity that BPE has never seen'
Tokens: ['Z', 'x', 'y', 'q', 'v', 'r', 'is', 'a', 'rare', 'ent', 'ity', 'that', 'B', 'P', 'E', 'has', 'never', 'seen']



In [ ]:
# Anchor question payoff: what happens to an OOV word?
oov_word = "Anthropomorphization"
enc = bpe_tokenizer.encode(oov_word)
print(f"OOV word: {oov_word!r}")
print(f"BPE splits it into subword pieces: {enc.tokens}")
print()
print("Key insight: BPE never produces [UNK] for a word — it decomposes into known subpieces.")
print("But what if the MEANING lives in the whole word (a rare named entity, a number)?")
print("→ The model sees broken subpieces, not the entity. Act 2's PGN solves this by COPYING.")

OOV word: 'Anthropomorphization'
BPE splits it into subword pieces: ['Anth', 'rop', 'om', 'or', 'ph', 'ization']

Key insight: BPE never produces [UNK] for a word — it decomposes into known subpieces.
But what if the MEANING lives in the whole word (a rare named entity, a number)?
→ The model sees broken subpieces, not the entity. Act 2's PGN solves this by COPYING.


### OOV and the Copy Problem

BPE's subword fallback is elegant but incomplete: when a rare named entity like *Quavrozymex* or a number like *$7.3B* appears in the source, the summarizer sees broken subpieces — and must *generate* something it has never seen as a unit.

The Pointer-Generator Network (Act 2) solves this directly: instead of generating the token, it **copies** it from the source via an attention distribution. That mechanism — a soft switch between *generate* and *copy* — is the core idea of See et al. 2017 (§2.1–2.2).

*(Vietnamese note: Vietnamese words are written with spaces between syllables, so BPE sometimes splits differently than a human would. A practitioner lesson, not a textbook one.)*

### 1b. BERT: MLM Demo + Head-Swap + Sentiment Fine-Tuning

The pretrain→finetune pattern has two steps:
1. **Pretrain** on a general task (masked-language modelling) to build a rich representation.
2. **Fine-tune** by attaching a small task-specific head and training on labelled data.

We make the head-swap **literal** — don't use `AutoModelForSequenceClassification` (it hides the swap). We build the head by hand so the transfer is visible.

*This MLM run is a toy — 20 steps on a tiny corpus, not real BERT pretraining (which takes GPU-days on a large corpus). The point is to see the body/head split.*

In [ ]:
import warnings
warnings.filterwarnings("ignore") # Ignore warning from `torch.cuda.amp.autocast`
!pip install --upgrade torch torchvision datasets

mlm_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
mlm = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")

# Tiny corpus: first 200 non-empty rows of wikitext-2 (toy MLM demo).
tiny = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:500]")
tiny = tiny.filter(lambda x: len(x["text"].strip()) > 20)

def tokenize_mlm(examples):
    return mlm_tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tiny_tok = tiny.map(tokenize_mlm, batched=True, remove_columns=["text"])
tiny_tok.set_format("torch")

data_collator = DataCollatorForLanguageModeling(tokenizer=mlm_tokenizer, mlm_probability=0.15)

mlm_args = TrainingArguments(
    output_dir="/tmp/mlm_output",
    num_train_epochs=1,
    max_steps=20,
    per_device_train_batch_size=8,
    logging_steps=5,
    report_to="none",
    use_cpu=True,
)
mlm_trainer = Trainer(
    model=mlm,
    args=mlm_args,
    train_dataset=tiny_tok,
    data_collator=data_collator,
)
mlm_trainer.train()
print("Toy MLM run complete.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Step,Training Loss
5,2.443632
10,2.576470
15,2.513067
20,2.174587


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Toy MLM run complete.


In [ ]:
# Reveal the body vs MLM-head split.
print("DistilBERT top-level modules:")
for name, _ in mlm.named_children():
    print(f"  {name}")
print()
print("'distilbert'  = pretrained BODY (the 6-layer transformer) → we keep this.")
print("Everything else = MLM HEAD (projects hidden states to ~30k vocab) → we drop this.")
print("We will replace the MLM head with a 2-class sentiment head.")

DistilBERT top-level modules:
  activation
  distilbert
  vocab_transform
  vocab_layer_norm
  vocab_projector
  mlm_loss_fct

'distilbert'  = pretrained BODY (the 6-layer transformer) → we keep this.
Everything else = MLM HEAD (projects hidden states to ~30k vocab) → we drop this.
We will replace the MLM head with a 2-class sentiment head.


In [ ]:
# TODO: implement SentimentModel
# Goal: attach a fresh classification head onto the pretrained DistilBERT body.
# body = pretrained transfer; head = random nn.Linear(body.config.dim, num_labels)
# forward: h = body(...).last_hidden_state → logits = head(h[:, 0])  ([CLS] = h[:,0])
# If labels given: loss = CrossEntropyLoss()(logits, labels)
# Return SequenceClassifierOutput(loss=loss, logits=logits)  (Trainer expects this)
from transformers.modeling_outputs import SequenceClassifierOutput
loss_fc = nn.CrossEntropyLoss()
class SentimentModel(nn.Module):
    def __init__(self, body, num_labels=2):
        super().__init__()
        self.body = body  # TODO: store the pretrained body
        self.head = nn.Linear(body.config.dim, num_labels)  # TODO: fresh nn.Linear(body.config.dim, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        h = self.body(input_ids, attention_mask).last_hidden_state
        logits = self.head(h[:,0])  # TODO: body → last_hidden_state → head([CLS] token)
        loss = None    # TODO: CrossEntropyLoss if labels is not None
        if labels is not None:
            loss = loss_fc(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)  # TODO

In [ ]:
sentiment_model = SentimentModel(mlm.distilbert)

body_params = sum(p.numel() for p in sentiment_model.body.parameters())
head_params  = sum(p.numel() for p in sentiment_model.head.parameters())
print(f"Body params (pretrained, transferred): {body_params:,}")
print(f"Head params (fresh, random):           {head_params:,}")        # 1538 since (768, 3) in sentiment head
print(f"Ratio: {body_params / head_params:.0f}x more params in body than head")
print()
print("That asymmetry IS transfer learning: millions of pretrained params doing heavy lifting,")
print("~1.5k random params doing the task-specific adaptation.")

Body params (pretrained, transferred): 66,362,880
Head params (fresh, random):           1,538
Ratio: 43149x more params in body than head

That asymmetry IS transfer learning: millions of pretrained params doing heavy lifting,
~1.5k random params doing the task-specific adaptation.


> **Thread #1 — pretrain→finetune (instance 1):** pretrain a general language representation, then attach a small task-specific head. The body weights carry knowledge from the pretraining corpus; only the head starts from scratch.
>
> We'll name this pattern again when we reach SFT in Act 3.

In [ ]:
import evaluate as hf_evaluate

# imdb = load_dataset("stanfordnlp/imdb", split="train[:500]")     #all 500 first are negatives, lol!
imdb = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(500))

def tokenize_imdb(examples):
    return mlm_tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

imdb_tok = imdb.map(tokenize_imdb, batched=True, remove_columns=["text"])
imdb_tok = imdb_tok.rename_column("label", "labels")
imdb_tok.set_format("torch")

ft_args = TrainingArguments(
    output_dir="/tmp/sentiment_output",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    logging_steps=10,
    report_to="none",
    use_cpu=True,
)

accuracy_metric = hf_evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy_metric.compute(predictions=logits.argmax(-1), references=labels)

ft_trainer = Trainer(
    model=sentiment_model,
    args=ft_args,
    train_dataset=imdb_tok,
    compute_metrics=compute_metrics,
)
ft_trainer.train()
print("Fine-tuning complete.")


#tokenizer -> pretrain -> custom sentiment head for finetune

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,0.678797
20,0.591240
30,0.516984


Fine-tuning complete.


### Bug 1 — Watch Carefully

We're about to run inference on a clearly positive review. There is a mistake in the code below.

Run it, observe the output, then diagnose together:
- Training loss went down ✓
- Predictions are wrong ✗

*Ask the class: why does training succeed but inference fail?*

In [ ]:
# BUG: using a mismatched tokenizer for inference.
# The model was trained with mlm_tokenizer (distilbert-base-uncased).
# Here we load a DIFFERENT tokenizer — watch what happens.
buggy_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")  # WRONG: cased ≠ uncased vocab

test_text = "This movie was absolutely fantastic, I loved every minute of it!"
inputs = buggy_tok(test_text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")

sentiment_model.eval()
with torch.no_grad():
    output = sentiment_model(inputs["input_ids"], inputs["attention_mask"])

pred = output.logits.argmax(-1).item()
print(f"Prediction: {'POSITIVE' if pred == 1 else 'NEGATIVE'}  (text is clearly positive)")
print(f"Logits: {output.logits.squeeze().tolist()}")
print()
print("Diagnose: what's different about the tokenizer we used?")

Prediction: POSITIVE  (text is clearly positive)
Logits: [-0.7621368765830994, 0.4341067373752594]

Diagnose: what's different about the tokenizer we used?


**Diagnosis:** `bert-base-cased` has a different vocabulary than `distilbert-base-uncased`. Token IDs from the wrong tokenizer map to completely different embeddings — garbage in, garbage out. Training used the *correct* tokenizer, so loss went down.

**The lesson:** tokenizer–model pairs are tightly coupled. Always load the tokenizer from the same checkpoint as the model. "Runs without error and loss goes down" does not mean "correct."

In [ ]:
# Fix: load the matching tokenizer.
correct_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

inputs_fixed = correct_tok(test_text, return_tensors="pt",
                           truncation=True, max_length=128, padding="max_length")

with torch.no_grad():
    output_fixed = sentiment_model(inputs_fixed["input_ids"], inputs_fixed["attention_mask"])

pred_fixed = output_fixed.logits.argmax(-1).item()
print(f"Prediction: {'POSITIVE' if pred_fixed == 1 else 'NEGATIVE'}")
print(f"Logits: {output_fixed.logits.squeeze().tolist()}")

Prediction: POSITIVE
Logits: [-0.7621368765830994, 0.4341067373752594]


### Wait — That Didn't Fix It

We fixed the tokenizer. The prediction is **still NEGATIVE** for *"This movie was absolutely fantastic, I loved every minute of it!"* — and the logits got *more* confident in the wrong direction:

```
Buggy tokenizer:   Logits ≈ [+2.9, -3.4]   (confidently negative)
Correct tokenizer: Logits ≈ [+4.0, -4.6]   (even more confidently negative)
```

Fixing Bug 1 made the model *more* wrong, not less. The tokenizer was a real bug — embeddings were garbage before — but it wasn't the *whole* bug. There's a deeper problem the tokenizer fix exposed.

*Ask the class: when a freshly fine-tuned classifier is **confidently wrong on every input**, what's the FIRST thing you should check?*

In [ ]:
from collections import Counter

# Diagnostic one-liner: always print the training label distribution.
label_counts = Counter(int(l) for l in imdb_tok["labels"])
print(f"Training label distribution: {dict(label_counts)}")
print(f"Total examples: {sum(label_counts.values())}")

Training label distribution: {1: 246, 0: 254}
Total examples: 500


**Diagnosis:** the entire training set is one class.

HuggingFace's `stanfordnlp/imdb` ships **sorted by label** — the first 12,500 rows are all negative, the next 12,500 are all positive. When we wrote `train[:500]`, we got 500 negatives and zero positives.

The model rationally learned the only signal available: *"always predict NEGATIVE, very confidently."* The loss curve went down because the loss is computed against negative labels — predicting NEGATIVE every time IS the optimal strategy for this (broken) training set. Loss going down didn't mean the model was learning sentiment; it meant the model was learning the constant function.

**The lesson — and it's a big one:**

> *Always print your label distribution before training.* One line — `Counter(labels)` — catches an entire class of silent bugs that no error message will ever warn you about.

This failure mode appears constantly in real systems:
- HuggingFace datasets sorted by label (this one)
- Time-sorted logs split as train/test → train sees only old data
- CSVs grouped by category
- Any pipeline where "the first N rows" feels safe but isn't

**Bug 1 was cosmetic** (wrong tokenizer → garbage logits). **Bug 1.5 is structural** (right tokenizer, right loss curve, wrong data). Fixing the visible bug revealed the deeper one. That happens *a lot* in practice.

In [ ]:
# Fix: shuffle BEFORE slicing so the 500-example subset is balanced.
imdb_balanced = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(500))
imdb_balanced_tok = imdb_balanced.map(tokenize_imdb, batched=True, remove_columns=["text"])
imdb_balanced_tok = imdb_balanced_tok.rename_column("label", "labels")
imdb_balanced_tok.set_format("torch")

print("Balanced label distribution:",
      dict(Counter(int(l) for l in imdb_balanced_tok["labels"])))

# Reinstantiate so the head starts fresh — otherwise it's still biased toward "always negative."
sentiment_model = SentimentModel(mlm.distilbert)

ft_trainer_fixed = Trainer(
    model=sentiment_model,
    args=ft_args,
    train_dataset=imdb_balanced_tok,
    compute_metrics=compute_metrics,
)
ft_trainer_fixed.train()
print("Re-finetuning on balanced data complete.")

Balanced label distribution: {1: 246, 0: 254}


Step,Training Loss
10,0.499875
20,0.375083
30,0.335468


Re-finetuning on balanced data complete.


In [ ]:
# Re-run inference with the same correct tokenizer.
with torch.no_grad():
    output_final = sentiment_model(inputs_fixed["input_ids"], inputs_fixed["attention_mask"])

pred_final = output_final.logits.argmax(-1).item()
print(f"Prediction: {'POSITIVE' if pred_final == 1 else 'NEGATIVE'}")
print(f"Logits: {output_final.logits.squeeze().tolist()}")
print()
print("Compare to before:")
print("  Bug 1 fix (sorted data):  Logits +4.0 / -4.6  → confidently NEGATIVE")
print("  Bug 1.5 fix (balanced):   Logits ~-0.7 / ~+0.6 → POSITIVE (low confidence — only 500 examples)")
print()
print("The model now leans the right way on this input. To get high confidence, train on more data")
print("(say train[:5000] balanced, or full IMDB on GPU). The point: it's now learning sentiment,")
print("not learning a constant.")

Prediction: POSITIVE
Logits: [-1.4935253858566284, 1.5768998861312866]

Compare to before:
  Bug 1 fix (sorted data):  Logits +4.0 / -4.6  → confidently NEGATIVE
  Bug 1.5 fix (balanced):   Logits ~-0.7 / ~+0.6 → POSITIVE (low confidence — only 500 examples)

The model now leans the right way on this input. To get high confidence, train on more data
(say train[:5000] balanced, or full IMDB on GPU). The point: it's now learning sentiment,
not learning a constant.


BERT *classifies* text — it understands. But what if the task is to **produce** text? A summary, a translation, an answer?

That requires a fundamentally different output space. To produce a sequence word by word, we need a **decoder**. Act 2 builds one from scratch — and the copy mechanism it uses is the conceptual precursor to everything in Act 3.

## Act 2 — Generate (Pointer-Generator Network)

**Anchor question:** What if the task requires producing a word the model has never seen in its vocabulary?

---

### Interlude: How Generation Models Are Trained

Before any PGN code, we need one concept that will echo through the rest of the tutorial.

**Teacher forcing.** At each decoding step $t$, feed the *gold* previous token $y_{t-1}$ (not the model's own prediction) and predict $y_t$. Cross-entropy over all steps. Fast, parallelizable, stable.

In code: `for t in range(Tdec): y_prev = tgt[:, t]  # gold token`.

**Exposure bias.** At inference the model feeds its *own* (sometimes wrong) outputs — a distribution it never saw during training. Train–test mismatch.

**Two forward-pointers:**
- *Scheduled sampling*: mix in the model's own predictions during training to soften the mismatch.
- *Optimize the metric you actually care about* (e.g. ROUGE) instead of token-level cross-entropy. **Hold this thought — it's exactly what GRPO does in Act 3.**

**Punchline (to cash in at Act 3a):** teacher forcing on `(input, target)` pairs *is* supervised fine-tuning. Act 3a will just put a name on it.

### PGN Architecture Walkthrough (See et al. 2017, §2.1–2.2)

We implement only the four equations that carry the core idea. Everything else is scaffolding.

**Encoder:** BiLSTM over source tokens → hidden states $h_i \in \mathbb{R}^{2H}$.

**Attention (eqs 1–3):**
$$e_i = v^\top \tanh(W_h h_i + W_s s_t) \quad \text{(score)}$$
$$a = \text{softmax}(e) \quad \text{(weights)}$$
$$\text{ctx} = \sum_i a_i h_i \quad \text{(context)}$$

**Vocab distribution (eq 4):**
$$P_{\text{vocab}} = \text{softmax}\!\left(V_{\text{proj}}([s_t \;; \text{ctx}])\right)$$

**Generation probability gate (eq 8):**
$$p_{\text{gen}} = \sigma(w \cdot [\text{ctx} \;; s_t \;; x_t])$$

**Mixed final distribution (eq 9) — THE CLIMAX:**
$$P_{\text{final}}(w) = p_{\text{gen}} \cdot P_{\text{vocab}}(w) + (1 - p_{\text{gen}}) \cdot \sum_{i:\, \text{src}_i = w} a_i$$

$p_{\text{gen}} \to 1$: generate from vocabulary. $p_{\text{gen}} \to 0$: copy from source.

In [ ]:
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


### 1. A Real Headline Dataset (Gigaword)

We use **Gigaword** — the canonical PGN-era benchmark for short-form summarization. Each example pairs a 1-sentence news article (~30 tokens) with a headline-length summary (~10 tokens). It's a real-world dataset, not synthetic. We load a 1,000-example subset so training stays CPU-friendly (~1 minute).

To make the copy mechanism *visible*, we deliberately **keep the vocabulary small** — only the ~1,500 most frequent tokens in the subset. Every named entity (company, person, country) and every number that's not common enough to make the cut becomes **out-of-vocabulary**. The PGN's only way to produce them in the headline is to **copy** from the article. $p_{\text{gen}}$ should drop toward 0 exactly on those tokens.

This is the same shape as the data the original PGN paper benchmarked on. Don't expect great ROUGE from a tiny CPU model — expect **decent**: short headline-shaped output that reuses the article's named entities and figures.

In [ ]:
import re
from collections import Counter

SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD, UNK, BOS, EOS = 0, 1, 2, 3

def simple_tokenize(text):
    # Words, numbers, and a small punctuation set. Lowercased.
    return re.findall(r"[a-z']+|\d+|[.,!?;:%$]", text.lower())

# Load a small Gigaword subset (real article→headline pairs).
gigaword = load_dataset("walzen/gigaword", split="train[:1000]")
print(f"Loaded {len(gigaword)} article-headline pairs from Gigaword.")

# Build a small vocab from the top-K most-frequent tokens.
# Small vocab → most named entities and numbers stay OOV → copy mechanism gets to shine.
TOP_K = 1500
counter = Counter()
for row in gigaword:
    counter.update(simple_tokenize(row["article"]))
    counter.update(simple_tokenize(row["summary"]))

FREQUENT = [w for w, _ in counter.most_common(TOP_K)]
ITOS = SPECIALS + FREQUENT
STOI = {w: i for i, w in enumerate(ITOS)}
V = len(ITOS)

# Pre-tokenize all examples once so training is fast.
examples = [(simple_tokenize(row["article"])[:50],   # cap article at 50 tokens
             simple_tokenize(row["summary"])[:15])    # cap headline at 15 tokens
            for row in gigaword]

# Show a few samples and count the OOV tokens in each headline
# (those are the tokens the model MUST copy — it cannot generate them).
print(f"\nFixed vocab size V={V}")
print(f"Total examples: {len(examples)}\n")
for i in [0, 1, 2]:
    src, tgt = examples[i]
    oov_in_tgt = [w for w in tgt if w not in STOI]
    print(f"ARTICLE : {' '.join(src)}")
    print(f"HEADLINE: {' '.join(tgt)}")
    print(f"  OOV tokens in headline (must be copied): {oov_in_tgt}\n")

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/296M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/296M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/29.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/299k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3803957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/189651 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1951 [00:00<?, ? examples/s]

Loaded 1000 article-headline pairs from Gigaword.

Fixed vocab size V=1504
Total examples: 1000

ARTICLE : officials of the cabinet level fair trade commission lrb ftc rrb said friday that they have formed an ad hoc group to investigate whether there is any manipulation of commodity prices by traders in local market .
HEADLINE: fair trade commission investigating consumer price hike
  OOV tokens in headline (must be copied): ['hike']

ARTICLE : five people were killed , and a woman gravely wounded , following a lethal shootout at a nightclub in cali , colombia 's third largest city , local authorities said monday .
HEADLINE: colombian nightclub shootout leaves five dead
  OOV tokens in headline (must be copied): ['colombian', 'nightclub']

ARTICLE : preliminary dna testing on remains of red army soldiers uncovered at a soviet era war memorial will be conducted in russia and ukraine , reports from the estonian capital of tallin said wednesday .
HEADLINE: estonia provides red army soldie

### 2. Numericalization with an Extended Vocab

Each example gets a **per-example extended vocab**: any source token outside the fixed vocab (a rare name, a number, a location) is assigned a temporary id $V, V+1, V+2, \ldots$ — *just for this one example.*

- `src_in`: encoder input — OOV tokens replaced with `<unk>` (the encoder's embedding table only knows the fixed vocab).
- `src_ext`: extended source ids — preserves OOV identity so the copy distribution knows where to scatter mass.
- `tgt_ext`: target uses the same per-example OOV ids, so the loss can reward correct copying.

The copy distribution scatters attention weights onto these extended ids; the generate distribution covers the fixed $V$ tokens. This is what makes "say a word the vocab doesn't contain" possible.

In [ ]:
def numericalize(src_tokens, tgt_tokens):
    oov2id, src_ext = {}, []
    for w in src_tokens:
        if w in STOI:
            src_ext.append(STOI[w])
        else:
            if w not in oov2id:
                oov2id[w] = V + len(oov2id)
            src_ext.append(oov2id[w])
    src_in = [i if i < V else UNK for i in src_ext]
    tgt_ext = []
    for w in tgt_tokens:
        if w in STOI:       tgt_ext.append(STOI[w])
        elif w in oov2id:   tgt_ext.append(oov2id[w])
        else:               tgt_ext.append(UNK)
    tgt_ext = [BOS] + tgt_ext + [EOS]
    return src_in, src_ext, tgt_ext, oov2id

#V=1500
# src = ["president", "clinton", "arrived", "in", "ireland"]
# oov = ["clinton": 1504, "ireland": 1505]
# src_in = [10, 1, 42, 8, 15, 1] -> unk -> 1
# src_extra = [10, 1504, 42, 8, 15, 1505]
# target_extra = [2, 1504, 7, 6, 8, 1505, 3]


from torch.nn.utils.rnn import pad_sequence

def batchify(examples):
    rows = [numericalize(s, t) for s, t in examples]
    # pad_sequence does the manual padding loop for us.
    src_in  = pad_sequence([torch.tensor(r[0]) for r in rows], batch_first=True, padding_value=PAD)
    src_ext = pad_sequence([torch.tensor(r[1]) for r in rows], batch_first=True, padding_value=PAD)
    tgt     = pad_sequence([torch.tensor(r[2]) for r in rows], batch_first=True, padding_value=PAD)
    src_len = torch.tensor([len(r[0]) for r in rows])
    max_oov = max(len(r[3]) for r in rows)
    src_toks = [s for s, _ in examples]
    return (src_in.to(DEVICE), src_ext.to(DEVICE), src_len.to(DEVICE),
            tgt.to(DEVICE), max_oov, src_toks)

## 3. Model

The full PGN class. Four student pieces live inside `decode_step` — everything else is scaffolding.

- **Encoder:** BiLSTM, bidirectional, hidden size $H=128$.
- **Decoder:** single-direction LSTMCell.
- **`decode_step`:** attention → vocab dist → $p_{\text{gen}}$ gate → mixed final dist.

In [ ]:
EMB, HID = 64, 128

class PGN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, EMB, padding_idx=PAD)
        self.enc = nn.LSTM(EMB, HID, batch_first=True, bidirectional=True)   #LSTM
        self.reduce_h = nn.Linear(2*HID, HID); self.reduce_c = nn.Linear(2*HID, HID)
        self.dec = nn.LSTMCell(EMB, HID)
        # attention params (eqs 1-3)
        self.W_h = nn.Linear(2*HID, HID, bias=False)
        self.W_s = nn.Linear(HID, HID)
        self.v   = nn.Linear(HID, 1, bias=False)
        # vocab projection (eq 4)
        self.Vproj = nn.Sequential(nn.Linear(HID + 2*HID, HID), nn.Linear(HID, V))
        # p_gen params (eq 8): from context, state, input
        self.w_pgen = nn.Linear(2*HID + HID + EMB, 1)

    def encode(self, src_in, src_len):
        # input_ids -> embedding vector
        e = self.emb(src_in)
        # embedding vector -> LSTM
        enc_out, (h,c) = self.enc(e)   #enc_out: B, S, 2H : hidden state for all tokens in sequence B, 2, 128
        h = torch.tanh(self.reduce_h(torch.cat([h[0],h[1]],-1)))
        c = torch.tanh(self.reduce_c(torch.cat([c[0],c[1]],-1)))
        return enc_out, (h,c)

    def attention(self, enc_out, s_t, src_mask):
        # >>> STUDENT PIECE 1 — attention, eqs (1)-(3) <<<
        # Goal: a (attention weights [B,S]) and ctx (context [B,2H]).
        # eq 1: e_i = v^T tanh(W_h h_i + W_s s_t)   score every source position
        enc_feat = self.W_h (enc_out)
        dec_feat = self.W_s (s_t)
        e = self.v (torch.tanh(enc_feat + dec_feat))    # B, S, H + B, 1, H -> Broadcast B, S, H -> B, S, 1

        # eq 2: a   = softmax(e)   (mask PAD with -1e9 BEFORE softmax)
        e = e.masked_fill(src_mask = 0, -1e9)
        a = F.softmax(e, dim=-1)
        # a: B, S    -> B, 1, S         enc_out: B, S, 2H     -> B, 1, 2H   -> B, 2H

        # eq 3: ctx = sum_i a_i * h_i
        #cant use torch.matmul, since would often collapse the 1 in B, 1, S
        ctx = torch.bmm(a.unsqueeze(1), enc_out).squeeze(1)   # to multiply these two matrices, have to unsqueeze
        # >>> END PIECE 1 <<<
        return a, ctx

    def decode_step(self, y_prev, state, enc_out, src_mask, src_ext, max_oov):
        x = self.emb(y_prev)                                 # [B,EMB]
        h,c = self.dec(x, state); s_t = h
        a, ctx = self.attention(enc_out, s_t, src_mask)

        # >>> STUDENT PIECE 2 — vocab distribution, eq (4) <<<
        # Goal: P_vocab over the FIXED vocab, shape [B,V].
        # eq 4: P_vocab = softmax( Vproj([s_t ; ctx]) )
        P_vocab = F.softmax(self.Vproj(torch.cat([s_t, ctx], -1)), dim = -1)   # Formula No.4
        # >>> END PIECE 2 <<<

        # >>> STUDENT PIECE 3 — generation probability gate, eq (8) <<<
        # Goal: p_gen in [0,1], shape [B,1].
        # eq 8: p_gen = sigmoid( w . [ctx ; s_t ; x] )
        p_gen = torch.sigmoid(self.w_pgen(torch.cat([ctx, s_t, x])))   # TODO
        # >>> END PIECE 3 <<<

        # >>> STUDENT PIECE 4 — mix into final distribution over extended vocab, eq (9) <<<
        # THE CLIMAX. Build P_final over the EXTENDED vocab, shape [B, V+max_oov]:
        #   P_final(w) = p_gen * P_vocab(w)  +  (1-p_gen) * sum_{i: src_word_i==w} a_i
        # Step A: zeros tensor of width V+max_oov.
        # Step B: put GENERATE mass (p_gen * P_vocab) into the first V columns.
        # Step C: SCATTER-ADD the COPY mass ((1-p_gen)*a) onto columns given by src_ext.
        B = P_vocab.size(0)    # batch_size
        P_final = torch.zeros(B, V+max_oov, device = P_vocab.device)   # TODO (Steps A-C)
        P_final[:,:V] = p_gen * P_vocab
        ext = ext_scatter_add(1, src_ext, (1 - p_gen) * a)
        # >>> END PIECE 4 <<<
        return P_final, p_gen.squeeze(-1), a, (h,c)

    def forward(self, src_in, src_ext, src_len, tgt, max_oov):
        enc_out, state = self.encode(src_in, src_len)
        src_mask = (src_in != PAD).long()
        Tdec = tgt.size(1) - 1
        loss = 0.0; ntok = 0
        for t in range(Tdec):
            y_prev = tgt[:, t].clone()                       # TEACHER FORCING: gold prev token
            y_prev[y_prev >= V] = UNK
            P_final, _, _, state = self.decode_step(y_prev, state, enc_out, src_mask, src_ext, max_oov)
            gold = tgt[:, t+1]
            mask = (gold != PAD).float()
            p = P_final.gather(1, gold.clamp(max=P_final.size(1)-1).unsqueeze(1)).squeeze(1)
            loss = loss - (torch.log(p + 1e-9) * mask).sum()
            ntok += mask.sum()
        return loss / ntok

## 4. Train (Teacher Forcing)

We sample minibatches from the pre-tokenized Gigaword `examples` list and run standard teacher-forced training. The `for t in range(Tdec)` loop feeds `tgt[:, t]` — the gold previous token — at each decoder step. That's teacher forcing.

Expect the loss to descend but **not to reach zero** — real data has irreducible variance (multiple valid headlines for one article, noise in the corpus, etc.). The current synthetic-data version reached ~0; real data plateaus around 2–3 with 600 CPU steps. That's the point — *summarization is not a clean memorization task.*

**Thread #2 checkpoint:** this loop structure *is* the SFT loop. Act 3a will put both side by side.

In [ ]:
def train(steps=600, bs=32, lr=1e-3):
    model = PGN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(1, steps + 1):
        batch_examples = random.sample(examples, bs)  # sample from the loaded Gigaword subset
        src_in, src_ext, src_len, tgt, max_oov, _ = batchify(batch_examples)
        loss = model(src_in, src_ext, src_len, tgt, max_oov)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
        if step % 100 == 0: print(f"step {step:4d}  loss {loss.item():.3f}")
    return model

## 5. Decode + Visualize p_gen Swing (the Payoff)

After training, decode a few articles greedily and record $p_{\text{gen}}$ and attention at each step. We also compute ROUGE-1 against the gold headline so we can put a number on "decent."

**What to look for:**
- Output reads like a *headline* — short, telegraphic, often grammatically rough but topical.
- ROUGE-1 typically lands in the **0.20–0.50 range** across examples (some examples will be lucky/unlucky). The model gets named entities and topic words right; it loses points on word order and on the words it `<unk>`s out.
- In the plot: $p_{\text{gen}}$ drops (red bars) when the decoder emits OOV tokens (named entities, numbers). The heatmap shows attention peaking on the source position of the token being copied.

*The visualization is the mechanism made visible — on real data this time.*

In [ ]:
@torch.no_grad()
def decode_and_record(model, example, max_steps=10):
    src, tgt = example
    src_in, src_ext, src_len, _, max_oov, src_toks = batchify([example])
    enc_out, state = model.encode(src_in, src_len)
    src_mask = (src_in != PAD).long()
    # id_to_word: real vocab + per-example OOV slots, all in one dict.
    oov2id = numericalize(src, tgt)[3]
    id_to_word = {i: w for i, w in enumerate(ITOS)}
    id_to_word.update({i: w for w, i in oov2id.items()})

    y = torch.tensor([BOS], device=DEVICE)
    out_tokens, pgens, attns = [], [], []
    for _ in range(max_steps):
        P_final, p_gen, a, state = model.decode_step(y, state, enc_out, src_mask, src_ext, max_oov)
        idx = P_final.argmax(-1).item()
        out_tokens.append(id_to_word.get(idx, "<unk>"))
        pgens.append(p_gen.item()); attns.append(a.squeeze(0).cpu().numpy())
        if idx == EOS: break
        y = torch.tensor([idx if idx < V else UNK], device=DEVICE)
    return src_toks[0], out_tokens, pgens, attns

def plot(src, out, pgens, attns):
    A = np.array(attns)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(7, len(src)*0.7), 6),
                                   gridspec_kw={"height_ratios": [3, 1]})
    ax1.imshow(A, aspect="auto", cmap="viridis")
    ax1.set_xticks(range(len(src))); ax1.set_xticklabels(src, rotation=45, ha="right")
    ax1.set_yticks(range(len(out))); ax1.set_yticklabels(out)
    ax1.set_title("Attention (rows = generated token, cols = source token)")
    ax2.bar(range(len(pgens)), pgens,
            color=["#d62728" if p<.5 else "#2ca02c" for p in pgens])
    ax2.axhline(.5, ls="--", c="gray", lw=.8)
    ax2.set_xticks(range(len(out))); ax2.set_xticklabels(out, rotation=45, ha="right")
    ax2.set_ylim(0, 1); ax2.set_ylabel("p_gen")
    ax2.set_title("p_gen per step  (red = COPYING from source, green = GENERATING from vocab)")
    plt.tight_layout(); plt.show()

In [ ]:
from rouge_score import rouge_scorer as rs
scorer = rs.RougeScorer(['rouge1'], use_stemmer=True)

pgn_model = train()

# Decode 3 random Gigaword articles. Score against the gold headline.
random.seed(0)
demo_examples = random.sample(examples, 3)
for ex in demo_examples:
    src_out, out_tokens, pgens, attns = decode_and_record(pgn_model, ex, max_steps=15)
    pred = " ".join(w for w in out_tokens if w != "<eos>")
    gold = " ".join(ex[1])
    rouge1 = scorer.score(gold, pred)['rouge1'].fmeasure
    print("ARTICLE :", " ".join(ex[0]))
    print("GOLD HL :", gold)
    print("PGN HL  :", pred)
    print(f"ROUGE-1 : {rouge1:.3f}")
    print()

# Plot p_gen + attention for the last decoded example (kept short for readability).
plot(src_out, out_tokens, pgens, attns)

AttributeError: 'NoneType' object has no attribute 'size'

### Bug 2 — Scatter-Add Misalignment

Here's a version a student wrote first. The `PGN` class above is correct — but this alternative patches in a broken `decode_step` with a one-position shift in the scatter-add index.

Run it, observe the heatmap, and diagnose together: loss still went down, but does the attention align with the right source tokens?

In [ ]:
import types

def _buggy_decode_step(self, y_prev, state, enc_out, src_mask, src_ext, max_oov):
    """Bug: scatter_add uses (src_ext + 1) — copy mass lands on the NEXT source token."""
    x = self.emb(y_prev)
    h, c = self.dec(x, state); s_t = h
    a, ctx = self.attention(enc_out, s_t, src_mask)
    P_vocab = F.softmax(self.Vproj(torch.cat([s_t, ctx], -1)), dim=-1)
    p_gen = torch.sigmoid(self.w_pgen(torch.cat([ctx, s_t, x], -1)))
    B = P_vocab.size(0)
    ext = torch.zeros(B, V + max_oov, device=P_vocab.device)
    ext[:, :V] = p_gen * P_vocab
    # BUG: shift source extended ids by +1 → copy mass lands on the wrong column
    src_ext_shifted = (src_ext + 1).clamp(max=V + max_oov - 1)
    ext = ext.scatter_add(1, src_ext_shifted, (1 - p_gen) * a)
    return ext, p_gen.squeeze(-1), a, (h, c)

# Temporarily patch decode_step with the buggy version.
pgn_model.decode_step = types.MethodType(_buggy_decode_step, pgn_model)

# Pick a short Gigaword example so the heatmap stays readable.
ex_bug = min(random.sample(examples, 5), key=lambda e: len(e[0]))
src_bug, out_bug, pgens_bug, attns_bug = decode_and_record(pgn_model, ex_bug, max_steps=15)
print("ARTICLE :", " ".join(ex_bug[0]))
print("GOLD HL :", " ".join(ex_bug[1]))
print("PGN HL  :", " ".join(out_bug))
plot(src_bug, out_bug, pgens_bug, attns_bug)

# Restore the correct decode_step (remove instance attribute → falls back to class method).
del pgn_model.decode_step

**Bug 2 diagnosis:** the scatter-add index was shifted by +1 — copy mass was scattered onto the *next* source token's column instead of the correct one. Loss still decreased because shifted scatter-add produces valid gradients. But the heatmap reveals the misalignment: attention peaks and copy payoff are one column off.

The correct version uses `src_ext` directly (no shift). The heatmap is the diagnostic tool — "loss went down" is not.

### BART Bridge (prose only)

PGN uses an **explicit copy mechanism**: $p_{\text{gen}}$ soft-switches between generate and copy, and scatter-add literally moves attention mass onto source token ids.

BART (and T5, mBART) do this **implicitly**: cross-attention in the decoder attends to encoder states, and a large shared vocabulary + pretraining makes OOV rare. The copy mechanism is absorbed into the weights.

**PGN is the white-box version of what BART does opaquely.** Once you understand the scatter-add, cross-attention is just a learned, continuous version of the same idea.

**Honesty caveat:** encoder–decoder models (T5, BART, mBART) are still SOTA-competitive and cheaper for production summarization and translation. We take the decoder-only path in Act 3 because it carries the alignment lesson — not because seq2seq is dead.